# Notebook 1 — Preparación de Datos: DisTEMIST (mT5 Pipeline)

**Objetivo:** Cargar el corpus clínico **DisTEMIST** (menciones de enfermedades y códigos en español) y estructurar los splits reproducibles `train` / `dev` / `test` a nivel de documento (`doc_id`).

### Lógica de Reutilización Compartida:
1. **Comprobación previa en Google Drive:** Si el split oficial `distemist_final/distemist_raw_splits` ya existe en Drive, **se carga directamente sin regenerar**, garantizando que el pipeline de mT5 consuma exactamente los mismos documentos que el resto del equipo.
2. **Generación en caso de ausencia:** Si no existe, se procesa desde los archivos crudos usando todos los documentos (`max_docs = None`), `seed = 42`, `train = 80%`, `dev = 10%`, `test = 10%` a nivel de documento y se guarda en Drive.

## 1. Conexión a Google Drive e Instalación de Dependencias

In [ ]:
# Montar Google Drive en entorno Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive montado exitosamente.")
except ImportError:
    print("ℹ️ Ejecutando en entorno local o fuera de Colab.")

In [ ]:
# Instalación de librerías para manejo de datasets
try:
    import datasets, huggingface_hub
    print("✅ Librerías disponibles.")
except ImportError:
    %pip install -q datasets huggingface_hub pandas
    print("✅ Librerías instaladas.")

In [ ]:
import json
import os
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from datasets import Dataset, DatasetDict, load_from_disk

# Fijar semilla global
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Rutas estándar en Google Drive para el proyecto
BASE_DIR = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1")
RAW_DATA_DIR = BASE_DIR / "distemist_raw" / "training"
FINAL_DATA_DIR = BASE_DIR / "distemist_final"
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Bandera para pruebas sintéticas fuera de línea
USE_MOCK_DATA = False

## 2. Detección de Splits Compartidos en Drive

In [ ]:
raw_splits_path = FINAL_DATA_DIR / "distemist_raw_splits"

if raw_splits_path.exists():
    print("=" * 65)
    print(" [OK] SPLIT OFICIAL DETECTADO EN GOOGLE DRIVE")
    print(f" Ruta: {raw_splits_path}")
    print("=" * 65)
    raw_splits = load_from_disk(str(raw_splits_path))
    for name, subset in raw_splits.items():
        print(f" Split {name:<6}: {len(subset):>4} documentos")
    print("\nSe utilizarán los splits existentes para mantener consistencia con el equipo.")
else:
    print("=" * 65)
    print(" [INFO] No se encontró split previo en Drive.")
    print(f" Se generarán los splits oficiales desde: {RAW_DATA_DIR}")
    print("=" * 65)

## 3. Funciones de Carga de Datos Crudos DisTEMIST (TSV y TXT)

In [ ]:
def fix_unicode_escapes(text: str) -> str:
    """Corrige secuencias unicode escapadas en las anotaciones."""
    return re.sub(r'\\u([0-9a-fA-F]{4})', lambda m: chr(int(m.group(1), 16)), text)

def load_from_tsv(raw_dir=RAW_DATA_DIR, txt_subdir="text_files",
                  tsv_dir="subtrack1_entities", max_docs=None, seed=SEED):
    """
    Carga los textos clínicos (.txt) y las entidades asociadas (.tsv) del corpus DisTEMIST.
    Conserva fielmente doc_id, offsets (start, end), texto de la entidad, label y código.
    """
    import csv
    import glob

    raw_dir = Path(raw_dir)
    txt_dir = raw_dir / txt_subdir
    tsv_paths = sorted(glob.glob(str(raw_dir / tsv_dir / "*.tsv")))

    if not tsv_paths:
        raise FileNotFoundError(f"No se encontraron archivos .tsv en {raw_dir / tsv_dir}")

    # Agrupar anotaciones por documento
    anns_by_doc = {}
    for tsv_path in tsv_paths:
        with open(tsv_path, encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                doc_id = row["filename"]
                span_text = fix_unicode_escapes(row["span"])
                anns_by_doc.setdefault(doc_id, []).append({
                    "start": int(row["off0"]),
                    "end": int(row["off1"]),
                    "text": span_text,
                    "label": row.get("label", "ENFERMEDAD"),
                    "code": row.get("codes", "")
                })

    txt_paths = sorted(txt_dir.glob("*.txt"))

    if max_docs is not None and max_docs < len(txt_paths):
        txt_paths = random.Random(seed).sample(txt_paths, max_docs)
        txt_paths = sorted(txt_paths)

    records = []
    for txt_path in txt_paths:
        doc_id = txt_path.stem
        text = txt_path.read_text(encoding="utf-8")
        entities = anns_by_doc.get(doc_id, [])
        records.append({"doc_id": doc_id, "text": text, "entities": entities})
        
    return records

In [ ]:
# Datos sintéticos de contingencia
MOCK_RECORDS = [
    {
        "doc_id": "mock_001",
        "text": "Paciente varón de 68 años que ingresa por disnea progresiva y tos con expectoracion. "
                "En la exploracion se objetiva neumonia adquirida en la comunidad. Antecedente de diabetes mellitus tipo 2.",
        "entities": [
            {"start": 108, "end": 143, "text": "neumonia adquirida en la comunidad", "label": "ENFERMEDAD", "code": "233604007"},
            {"start": 166, "end": 189, "text": "diabetes mellitus tipo 2", "label": "ENFERMEDAD", "code": "44054006"},
        ],
    },
    {
        "doc_id": "mock_002",
        "text": "Mujer de 45 años con antecedentes de hipertension arterial que consulta por cefalea intensa "
                "de una semana de evolucion, sin fiebre asociada.",
        "entities": [
            {"start": 38, "end": 59, "text": "hipertension arterial", "label": "ENFERMEDAD", "code": "38341003"},
        ],
    },
]

if not raw_splits_path.exists():
    if USE_MOCK_DATA:
        raw_records = MOCK_RECORDS
    else:
        # max_docs=None para procesar todos los documentos disponibles
        raw_records = load_from_tsv(max_docs=None)
    print(f"Total de documentos cargados: {len(raw_records)}")

## 4. Partición a Nivel de Documento (`doc_id`) y Guardado en Drive

In [ ]:
def make_split(records, train_ratio=0.8, dev_ratio=0.1, seed=SEED):
    """Genera splits train/dev/test a nivel de documento completo."""
    records = records[:]
    random.Random(seed).shuffle(records)
    n = len(records)
    n_train = max(1, int(n * train_ratio))
    n_dev = max(1, int(n * dev_ratio))
    return {
        "train": records[:n_train],
        "dev": records[n_train:n_train + n_dev],
        "test": records[n_train + n_dev:],
    }

if not raw_splits_path.exists():
    splits = make_split(raw_records, train_ratio=0.8, dev_ratio=0.1, seed=SEED)
    raw_splits = DatasetDict({
        split_name: Dataset.from_list(records)
        for split_name, records in splits.items()
    })
    raw_splits.save_to_disk(str(raw_splits_path))
    print(f"✅ Splits guardados exitosamente en: {raw_splits_path}")
else:
    print(f"ℹ️ Splits ya presentes en Drive. No se sobrescribieron.")

## 5. Validación de Independencia y No Solapamiento

In [ ]:
train_ids = set(raw_splits["train"]["doc_id"])
dev_ids = set(raw_splits["dev"]["doc_id"])
test_ids = set(raw_splits["test"]["doc_id"])

assert len(train_ids & dev_ids) == 0, "¡Fuga detectada entre train y dev!"
assert len(train_ids & test_ids) == 0, "¡Fuga detectada entre train y test!"
assert len(dev_ids & test_ids) == 0, "¡Fuga detectada entre dev y test!"

print("=" * 60)
print("  VALIDACIÓN DE INDEPENDENCIA DE SPLITS")
print("=" * 60)
print(f" Documentos en Train : {len(train_ids)}")
print(f" Documentos en Dev   : {len(dev_ids)}")
print(f" Documentos en Test  : {len(test_ids)}")
print(f" Total Documentos    : {len(train_ids) + len(dev_ids) + len(test_ids)}")
print(f" Intersección cruzada: 0 documentos")
print("=" * 60)
print("✅ Datos preparados correctamente. Continuar con Notebook 02.")